In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')

# Ensure timestamp numeric
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')

# --- metric.csv summaries ---
# unique cmdb_id list (count + top 20)
metric_cmdb_all = sorted(metric_df['cmdb_id'].dropna().unique().tolist())
metric_cmdb_count = len(metric_cmdb_all)
metric_cmdb_unique_top20 = pd.DataFrame(metric_cmdb_all[:20], columns=['cmdb_id'])

# kpi_name counts (top 20)
metric_kpi_counts = metric_df.groupby('kpi_name').size().sort_values(ascending=False).head(20).reset_index(name='count')

# min / max timestamp with UTC+8 conversion
if metric_df['timestamp'].dropna().shape[0] > 0:
    metric_min_ts = int(metric_df['timestamp'].min())
    metric_max_ts = int(metric_df['timestamp'].max())
    metric_min_local = pd.to_datetime(metric_min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    metric_max_local = pd.to_datetime(metric_max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    metric_min_ts = metric_max_ts = None
    metric_min_local = metric_max_local = None

metric_ts_summary = pd.DataFrame([{
    'min_epoch': metric_min_ts,
    'min_local_UTC+8': metric_min_local,
    'max_epoch': metric_max_ts,
    'max_local_UTC+8': metric_max_local
}])

# first up to 20 rows
metric_head20 = metric_df.head(20)

# --- trace.csv summaries ---
# unique cmdb_id list (count + top 20)
trace_cmdb_all = sorted(trace_df['cmdb_id'].dropna().unique().tolist())
trace_cmdb_count = len(trace_cmdb_all)
trace_cmdb_unique_top20 = pd.DataFrame(trace_cmdb_all[:20], columns=['cmdb_id'])

# trace_name counts (top 20)
trace_name_counts = trace_df.groupby('trace_name').size().sort_values(ascending=False).head(20).reset_index(name='count')

# min / max timestamp with UTC+8 conversion
if trace_df['timestamp'].dropna().shape[0] > 0:
    trace_min_ts = int(trace_df['timestamp'].min())
    trace_max_ts = int(trace_df['timestamp'].max())
    trace_min_local = pd.to_datetime(trace_min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    trace_max_local = pd.to_datetime(trace_max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
else:
    trace_min_ts = trace_max_ts = None
    trace_min_local = trace_max_local = None

trace_ts_summary = pd.DataFrame([{
    'min_epoch': trace_min_ts,
    'min_local_UTC+8': trace_min_local,
    'max_epoch': trace_max_ts,
    'max_local_UTC+8': trace_max_local
}])

# first up to 20 rows
trace_head20 = trace_df.head(20)

# Compact summary variables to display (bounded)
metric_cmdb_count, metric_cmdb_unique_top20, metric_kpi_counts, metric_ts_summary, metric_head20, trace_cmdb_count, trace_cmdb_unique_top20, trace_name_counts, trace_ts_summary, trace_head20

```
Out[1]:
```
```python
# Summarize previous execution results in plain English (compact)
metric_top_cmdb = ', '.join(metric_cmdb_unique_top20['cmdb_id'].tolist())
metric_kpis_top = ', '.join([f"{row.kpi_name}({row['count']})" for row in metric_kpi_counts.itertuples()])
metric_sample_rows = []
for i in range(min(2, len(metric_head20))):
    r = metric_head20.iloc[i]
    metric_sample_rows.append(f"Row{i}: ts={int(r.timestamp)}, cmdb_id={r.cmdb_id}, kpi_name={r.kpi_name}, value={r.value}")
metric_sample_text = ' | '.join(metric_sample_rows)

trace_top_cmdb = ', '.join(trace_cmdb_unique_top20['cmdb_id'].tolist())
trace_names_top = ', '.join([f"{row.trace_name}({row['count']})" for row in trace_name_counts.itertuples()])
trace_sample_rows = []
for i in range(min(2, len(trace_head20))):
    r = trace_head20.iloc[i]
    trace_sample_rows.append(f"Row{i}: ts={int(r.timestamp)}, cmdb_id={r.cmdb_id}, trace_name={r.trace_name}, value={r.value}")
trace_sample_text = ' | '.join(trace_sample_rows)

summary = (
    "metric.csv summary:\n"
    f"- Unique cmdb_id count: {metric_cmdb_count}\n"
    f"- Up to 20 cmdb_id examples: {metric_top_cmdb}\n"
    f"- Top 20 kpi_name by count: {metric_kpis_top}\n"
    f"- Timestamp range (UTC+8): {metric_min_local} to {metric_max_local}\n"
    f"- Sample rows (first 2 of up to 20 shown): {metric_sample_text}\n\n"
    "trace.csv summary:\n"
    f"- Unique cmdb_id count: {trace_cmdb_count}\n"
    f"- Up to 20 cmdb_id examples: {trace_top_cmdb}\n"
    f"- Top trace_name by count (top 20 shown): {trace_names_top}\n"
    f"- Timestamp range (UTC+8): {trace_min_local} to {trace_max_local}\n"
    f"- Sample rows (first 2 of up to 20 shown): {trace_sample_text}\n"
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(56,        cmdb_id
0       db_001
1       db_002
2       db_003
3       db_004
4       db_005
5       db_006
6       db_007
7       db_008
8       db_009
9       db_010
10      db_011
11      db_012
12      db_013
13  docker_001
14  docker_002
15  docker_003
16  docker_004
17  docker_005
18  docker_006
19  docker_007,                     kpi_name  count
0                 Send_total   1321
1               Disk_io_util   1321
2   Incoming_network_traffic   1320
3                Disk_rd_ios   1319
4   Outgoing_network_traffic   1319
5            Memory_used_pct   1318
6           Memory_available   1318
7                Memory_used   1318
8                 Disk_svctm   1316
9                Disk_wr_kbs   1316
10                Recv_total   1316
11             CPU_user_time   1316
12               Disk_rd_kbs   1315
13               Disk_wr_ios   1315
14           CPU_iowait_time   1315
15           CPU_system_time   1315
16      Processor_load_5_min   1314
17                   Page_po   1312
18             Disk_avgqu_sz   1312
19                Agent_ping   1312,     min_epoch          min_local_UTC+8   max_epoch          max_local_UTC+8
0  1590775200  2020-05-30 02:00:00 CST  1590778800  2020-05-30 03:00:00 CST,      timestamp cmdb_id               kpi_name       value
0   1590775200  db_001                    ACS    0.010000
1   1590775200  db_001                   AIOS    0.000000
2   1590775200  db_001                    AWS    0.020000
3   1590775200  db_001           CPU_Used_Pct    1.391597
4   1590775200  db_001           CPU_free_pct   98.608403
5   1590775200  db_001           Call_Per_Sec   11.500000
6   1590775200  db_001    DFParaWrite_Per_Sec    0.000000
7   1590775200  db_001                 DbTime   14.500000
8   1590775200  db_001           Exec_Per_Sec   10.600000
9   1590775200  db_001                   Hang    1.000000
10  1590775200  db_001    LFParaWrite_Per_Sec    1.640000
11  1590775200  db_001         LFSync_Per_Sec    2.010000
12  1590775200  db_001     Logic_Read_Per_Sec    8.100000
13  1590775200  db_001          Login_Per_Sec    0.200000
14  1590775200  db_001              MEM_Total  381.902264
15  1590775200  db_001               MEM_Used  245.779535
16  1590775200  db_001           MEM_Used_Pct   64.356788
17  1590775200  db_001          MEM_real_util   46.220000
18  1590775200  db_001           On_Off_State    1.000000
19  1590775200  db_001  Physical_Read_Per_Sec    0.000000, 10,       cmdb_id
0  docker_001
1  docker_002
2  docker_003
3  docker_004
4  docker_005
5  docker_006
6  docker_007
7  docker_008
8      os_021
9      os_022,                                        trace_name  count
0                       trace.self.jdbc.row_count    228
1                      trace.self.jdbc.error_rate    228
2                    trace.self.jdbc.duration_p95    228
3                   trace.self.jdbc.duration_mean    228
4                      trace.self.local.row_count    228
5                     trace.self.local.error_rate    228
6                   trace.self.local.duration_p95    228
7                  trace.self.local.duration_mean    228
8                        trace.self.csf.row_count    172
9                       trace.self.csf.error_rate    172
10                    trace.self.csf.duration_p95    172
11                   trace.self.csf.duration_mean    172
12             trace.self.flyremote.duration_mean    120
13              trace.self.flyremote.duration_p95    120
14                trace.self.flyremote.error_rate    120
15                 trace.self.flyremote.row_count    120
16  trace.from_os_022.remoteprocess.duration_mean     60
17   trace.from_os_022.remoteprocess.duration_p95     60
18     trace.from_os_022.remoteprocess.error_rate     60
19      trace.from_os_022.remoteprocess.row_count     60,     min_epoch          min_local_UTC+8   max_epoch          max_local_UTC+8
0  1590777000  2020-05-30 02:30:00 CST  1590778740  2020-05-30 02:59:00 CST,      timestamp     cmdb_id                                     trace_name        value
0   1590777000  docker_001  trace.from_os_022.remoteprocess.duration_mean     0.711888
1   1590777000  docker_001   trace.from_os_022.remoteprocess.duration_p95     1.621000
2   1590777000  docker_001     trace.from_os_022.remoteprocess.error_rate     0.000000
3   1590777000  docker_001      trace.from_os_022.remoteprocess.row_count   143.000000
4   1590777000  docker_001                   trace.self.csf.duration_mean     0.062355
5   1590777000  docker_001                    trace.self.csf.duration_p95     0.124450
6   1590777000  docker_001                      trace.self.csf.error_rate     0.000000
7   1590777000  docker_001                       trace.self.csf.row_count   572.000000
8   1590777000  docker_001             trace.self.flyremote.duration_mean     0.010594
9   1590777000  docker_001              trace.self.flyremote.duration_p95     0.078500
10  1590777000  docker_001                trace.self.flyremote.error_rate     0.000000
11  1590777000  docker_001                 trace.self.flyremote.row_count   143.000000
12  1590777000  docker_001                  trace.self.jdbc.duration_mean     0.003951
13  1590777000  docker_001                   trace.self.jdbc.duration_p95     0.003000
14  1590777000  docker_001                     trace.self.jdbc.error_rate     0.000000
15  1590777000  docker_001                      trace.self.jdbc.row_count  2294.000000
16  1590777000  docker_001                 trace.self.local.duration_mean     0.017906
17  1590777000  docker_001                  trace.self.local.duration_p95     0.094000
18  1590777000  docker_001                    trace.self.local.error_rate     0.000000
19  1590777000  docker_001                     trace.self.local.row_count  2002.000000)```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

# reuse metric_df and tz if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full data
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_P95=lambda x: x.quantile(0.95),
         global_P5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start_ts = 1590777000
end_ts   = 1590778800
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# If no data in window, return explicit message
if window_df.shape[0] == 0:
    result = "No metric data found in the incident window 1590777000..1590778800."
else:
    # 3) Merge thresholds into window and detect anomalies
    w = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

    # Compute anomaly count per pair in window
    agg_counts = (
        w
        .groupby(['cmdb_id','kpi_name'])['is_anomaly']
        .sum()
        .reset_index(name='anomaly_count_in_window')
    )

    # earliest anomaly timestamp per pair (epoch)
    earliest = (
        w[w['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'])['timestamp']
        .min()
        .reset_index(name='earliest_anomaly_epoch')
    )

    # max/min value in window per pair
    minmax = (
        w
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(max_value_in_window='max', min_value_in_window='min')
        .reset_index()
    )

    # Merge computed pieces with global thresholds
    result_df = (
        agg_counts
        .merge(earliest, on=['cmdb_id','kpi_name'], how='left')
        .merge(minmax, on=['cmdb_id','kpi_name'], how='left')
        .merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    )

    # Format earliest anomaly timestamp to UTC+8 local string; keep NaN if no anomaly
    def fmt_epoch_to_local_s(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    result_df['earliest_anomaly_timestamp'] = result_df['earliest_anomaly_epoch'].apply(fmt_epoch_to_local_s)

    # Round numeric thresholds/values for compactness
    for col in ['global_P95','global_P5','max_value_in_window','min_value_in_window']:
        if col in result_df.columns:
            result_df[col] = result_df[col].round(6)

    final_cols = [
        'cmdb_id','kpi_name','global_P95','global_P5',
        'anomaly_count_in_window','earliest_anomaly_timestamp',
        'max_value_in_window','min_value_in_window'
    ]
    result_df = result_df[final_cols]

    # 4) Sort by anomaly_count_in_window desc and limit to top 20
    result_df = result_df.sort_values(by='anomaly_count_in_window', ascending=False).reset_index(drop=True)

    # If no anomalies at all, return explicit message
    if result_df['anomaly_count_in_window'].sum() == 0:
        result = "No anomalies detected in the incident window based on global P95/P5 thresholds."
    else:
        result = result_df.head(20)

# Display result (DataFrame or message)
result

```
Out[2]:
```
```python
# Summarize the previously displayed anomaly detection results in plain English.
try:
    df = result  # reuse result from prior computation (top 20 rows)
except NameError:
    df = None

if df is None or (isinstance(df, str) and df.startswith("No")):
    summary = "No anomaly results available to summarize."
else:
    total_pairs_shown = int(df.shape[0])
    total_anomalies_shown = int(df['anomaly_count_in_window'].sum())
    top3 = [f"{r.cmdb_id}/{r.kpi_name}({int(r.anomaly_count_in_window)})" for r in df.head(3).itertuples(index=False)]
    earliest_times = df['earliest_anomaly_timestamp'].dropna().unique().tolist()
    earliest_sample = earliest_times[:3]
    # Common observation about values (many rows show constant max=min or zeros)
    const_value_count = int((df['max_value_in_window'] == df['min_value_in_window']).sum())
    zero_value_count = int(((df['max_value_in_window'] == 0) & (df['min_value_in_window'] == 0)).sum())

    summary = (
        "Anomaly detection summary (incident window 2020-05-30 02:30 to 03:00 UTC+8):\n"
        f"- Showing top {total_pairs_shown} (sorted by anomaly_count). Total anomaly points in these rows: {total_anomalies_shown}.\n"
        f"- Top 3 affected metric pairs: {', '.join(top3)}.\n"
        f"- Earliest anomaly timestamps (examples): {', '.join(earliest_sample)}.\n"
        f"- {const_value_count} of the shown pairs have identical max/min in-window values (constant during window).\n"
        f"- {zero_value_count} of the shown pairs are constant zeros in the window.\n"
        "- Interpretation: Many metrics hit their global P95 or P5 repeatedly across the 30-minute window, often at the window start (02:30). This suggests sustained anomalous values (often constant or boundary values) for those metric pairs during the incident window."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                   kpi_name    global_P95     global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0       os_020               Disk_io_util  0.000000e+00  0.000000e+00                       31    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
1    redis_009                 redis_ping  1.000000e+00  1.000000e+00                       31    2020-05-30 02:30:00 CST         1.000000e+00         1.000000e+00
2       os_007                Disk_rd_ios  0.000000e+00  0.000000e+00                       31    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
3       db_009               On_Off_State  1.000000e+00  1.000000e+00                       30    2020-05-30 02:30:00 CST         1.000000e+00         1.000000e+00
4    redis_005       rejected_connections  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
5    redis_005                 redis_ping  1.000000e+00  1.000000e+00                       30    2020-05-30 02:30:00 CST         1.000000e+00         1.000000e+00
6    redis_005                 redis_load  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
7    redis_005                  maxmemory  2.000000e+09  2.000000e+09                       30    2020-05-30 02:30:00 CST         2.000000e+09         2.000000e+09
8    redis_005            keyspace_misses  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
9    redis_005              keyspace_hits  2.000000e+00  2.000000e+00                       30    2020-05-30 02:30:00 CST         2.000000e+00         2.000000e+00
10  docker_007             container_fgct  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
11  docker_006         container_mem_used  5.900000e+01  5.900000e+01                       30    2020-05-30 02:30:00 CST         5.900000e+01         5.900000e+01
12  docker_007  container_thread_used_pct  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
13  docker_007     container_thread_total  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
14  docker_007   container_thread_running  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
15      db_009        DFParaWrite_Per_Sec  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
16  docker_007     container_session_used  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
17  docker_007         container_mem_used  5.900000e+01  5.900000e+01                       30    2020-05-30 02:30:00 CST         5.900000e+01         5.900000e+01
18  docker_007              container_fgc  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00
19  docker_006  container_thread_used_pct  0.000000e+00  0.000000e+00                       30    2020-05-30 02:30:00 CST         0.000000e+00         0.000000e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz

# Reuse existing variables if present, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# timezone UTC+8
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series (must be before window filtering)
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_P95=lambda x: x.quantile(0.95),
         global_P5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window (inclusive)
start_ts = 1590777000
end_ts   = 1590778800
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

if window_df.shape[0] == 0:
    result = "No metric data found in the incident window 1590777000..1590778800."
else:
    # 3) Merge thresholds and flag anomalies in window
    w = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

    # Compute anomaly counts per (cmdb_id, kpi_name) in-window (include zero counts)
    per_kpi = (
        w
        .groupby(['cmdb_id','kpi_name'])
        .agg(anomaly_count=('is_anomaly','sum'))
        .reset_index()
    )

    # 4) Aggregate per cmdb_id
    # total anomaly points across kpis
    total_anom = per_kpi.groupby('cmdb_id')['anomaly_count'].sum().reset_index(name='total_anomaly_points')

    # distinct kpi count with anomalies (>0)
    distinct_kpi = (
        per_kpi[per_kpi['anomaly_count']>0]
        .groupby('cmdb_id')['kpi_name']
        .nunique()
        .reset_index(name='distinct_kpi_count_with_anomalies')
    )

    # earliest anomaly timestamp per cmdb_id (from anomaly rows)
    earliest = (
        w[w['is_anomaly']]
        .groupby('cmdb_id')['timestamp']
        .min()
        .reset_index(name='earliest_anomaly_epoch')
    )

    # max/min value overall in window per cmdb_id
    minmax = (
        window_df
        .groupby('cmdb_id')['value']
        .agg(max_value_overall_in_window='max', min_value_overall_in_window='min')
        .reset_index()
    )

    # 5) For sample kpis per cmdb_id: top up to 5 (kpi_name, anomaly_count) pairs ordered desc
    def make_sample_series(subdf):
        sub = subdf.sort_values('anomaly_count', ascending=False).head(5)
        pairs = [f"{row.kpi_name}({int(row.anomaly_count)})" for row in sub.itertuples(index=False)]
        return ", ".join(pairs) if pairs else pd.NA

    sample_df = (
        per_kpi
        .groupby('cmdb_id')
        .apply(make_sample_series)
        .reset_index(name='sample_kpis_with_counts')
    )

    # Merge all per-cmdb pieces
    cmdb_agg = (
        total_anom
        .merge(distinct_kpi, on='cmdb_id', how='left')
        .merge(earliest, on='cmdb_id', how='left')
        .merge(minmax, on='cmdb_id', how='left')
        .merge(sample_df, on='cmdb_id', how='left')
    )

    # Fill NaN distinct_kpi_count_with_anomalies with 0
    cmdb_agg['distinct_kpi_count_with_anomalies'] = cmdb_agg['distinct_kpi_count_with_anomalies'].fillna(0).astype(int)

    # Format earliest anomaly epoch to UTC+8 string
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    cmdb_agg['earliest_anomaly_timestamp'] = cmdb_agg['earliest_anomaly_epoch'].apply(fmt_epoch)

    # Round numeric max/min for compactness
    cmdb_agg['max_value_overall_in_window'] = cmdb_agg['max_value_overall_in_window'].round(6)
    cmdb_agg['min_value_overall_in_window'] = cmdb_agg['min_value_overall_in_window'].round(6)

    # Select and order columns
    final_cols = [
        'cmdb_id','total_anomaly_points','distinct_kpi_count_with_anomalies',
        'earliest_anomaly_timestamp','max_value_overall_in_window','min_value_overall_in_window',
        'sample_kpis_with_counts'
    ]
    cmdb_summary = cmdb_agg[final_cols]

    # 6) Sort by total_anomaly_points desc and limit to top 20
    cmdb_summary_top20 = cmdb_summary.sort_values('total_anomaly_points', ascending=False).reset_index(drop=True).head(20)

    # If total anomalies overall is zero, return explicit message
    if cmdb_summary_top20['total_anomaly_points'].sum() == 0:
        result = "No anomalies detected in the incident window based on global P95/P5 thresholds."
    else:
        result = cmdb_summary_top20

# Display final result (DataFrame or message)
result

```
Out[3]:
```
```python
# Summarize the cmdb-level anomaly aggregation (plain English) using prior result variable.
try:
    df = result  # reuse result from previous computation (top 20 cmdb_id rows)
except NameError:
    df = None

if df is None or isinstance(df, str):
    summary = "No aggregated anomaly results available."
else:
    # Top 5 cmdb_ids by total anomalies
    top5 = []
    for r in df.head(5).itertuples(index=False):
        top5.append(f"{r.cmdb_id}({int(r.total_anomaly_points)})")
    # Earliest anomaly timestamps (most are the same)
    earliest_examples = df['earliest_anomaly_timestamp'].dropna().unique()[:3].tolist()
    # Counts of cmdb_ids and total anomalies in shown rows
    shown_rows = int(df.shape[0])
    total_anom_top = int(df['total_anomaly_points'].sum())
    # How many distinct KPIs implicated on average (approx)
    avg_distinct_kpis = float(df['distinct_kpi_count_with_anomalies'].mean())
    # Note common patterns in sample_kpis strings (many show KPI(30) etc.)
    sample_examples = df['sample_kpis_with_counts'].dropna().head(3).tolist()

    summary = (
        "Incident window: 2020-05-30 02:30 to 03:00 (UTC+8).\n"
        f"- Top {min(5, len(top5))} cmdb_ids by anomaly points: {', '.join(top5)}.\n"
        f"- Showing {shown_rows} cmdb_id rows (top 20). Total anomaly points in these rows: {total_anom_top}.\n"
        f"- Earliest anomaly timestamps (examples): {', '.join(earliest_examples)}.\n"
        f"- On average these cmdb_ids had ~{avg_distinct_kpis:.1f} distinct KPIs with anomalies.\n"
        "- Many KPIs are flagged for nearly every minute in the window (sample entries show KPI(30) repeatedly),\n"
        "  and many metrics are constant during the window (max == min, often 0). This indicates widespread,\n"
        "  sustained anomalous values across many hosts/services beginning at the window start (02:30).\n"
        "- Actionable note: investigate common infrastructure/services (DB nodes and several OS/docker hosts listed at top)\n"
        "  and look for systemic causes (network, resource exhaustion, or service-wide configuration issues)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  total_anomaly_points  distinct_kpi_count_with_anomalies earliest_anomaly_timestamp  max_value_overall_in_window  min_value_overall_in_window                            sample_kpis_with_counts
0      db_006                   748                                 43    2020-05-30 02:30:00 CST                 2.433360e+04                          0.0  AIOS(30), Hang(30), TPS_Per_Sec(30), Row_Lock(...
1      db_013                   631                                 44    2020-05-30 02:30:00 CST                 2.983500e+04                          0.0  ACS(30), AIOS(30), AWS(30), Hang(30), Login_Pe...
2      os_007                   619                                 51    2020-05-30 02:30:00 CST                 4.360640e+05                          0.0  Disk_rd_ios(31), Disk_await(30), Agent_ping(30...
3      db_010                   615                                 45    2020-05-30 02:30:00 CST                 6.441400e+03                          0.0  ACS(30), AIOS(30), AWS(30), Hang(30), Login_Pe...
4      db_002                   599                                 43    2020-05-30 02:30:00 CST                 2.466270e+04                          0.0  ACS(30), AIOS(30), Hang(30), Row_Lock(30), Sct...
5      os_006                   595                                 50    2020-05-30 02:30:00 CST                 8.370800e+04                          0.0  Agent_ping(30), Disk_svctm(30), Disk_io_util(3...
6      db_005                   588                                 43    2020-05-30 02:30:00 CST                 2.346660e+04                          0.0  AIOS(30), Hang(30), Login_Per_Sec(30), Row_Loc...
7      db_008                   577                                 43    2020-05-30 02:30:00 CST                 2.232050e+04                          0.0  AIOS(30), Hang(30), MEM_Total(30), Row_Lock(30...
8      os_008                   575                                 51    2020-05-30 02:30:00 CST                 4.360640e+05                          0.0  Agent_ping(30), Disk_wr_ios(30), Disk_io_util(...
9      db_011                   574                                 43    2020-05-30 02:30:00 CST                 5.451100e+03                          0.0  ACS(30), AIOS(30), AWS(30), Hang(30), MEM_Tota...
10     db_001                   564                                 43    2020-05-30 02:30:00 CST                 2.686060e+04                          0.0  ACS(30), AIOS(30), Hang(30), Row_Lock(30), Sct...
11     db_007                   545                                 44    2020-05-30 02:30:00 CST                 3.177940e+04                          0.0  AIOS(30), Hang(30), Sess_Active(30), Row_Lock(...
12     db_012                   543                                 45    2020-05-30 02:30:00 CST                 1.741380e+04                          0.0  ACS(30), AIOS(30), AWS(30), Hang(30), MEM_Tota...
13  redis_004                   539                                 21    2020-05-30 02:30:00 CST                 8.000000e+09                          0.0  Redis_key_count(30), blocked_clients(30), conn...
14     db_003                   534                                 46    2020-05-30 02:30:00 CST                 1.135357e+05                          0.0  ACS(30), Hang(30), Row_Lock(30), On_Off_State(...
15  redis_012                   534                                 22    2020-05-30 02:30:00 CST                 8.000000e+09                          0.0  Redis_key_count(30), blocked_clients(30), conn...
16  redis_008                   532                                 20    2020-05-30 02:30:00 CST                 8.000000e+09                          0.0  Redis_key_count(30), blocked_clients(30), conn...
17     os_021                   530                                 49    2020-05-30 02:30:00 CST                 1.313000e+03                          0.0  Agent_ping(30), Disk_wr_ios(30), Disk_rd_kbs(3...
18     db_004                   529                                 43    2020-05-30 02:30:00 CST                 6.718322e+05                          0.0  ACS(30), AIOS(30), Hang(30), On_Off_State(30),...
19     os_014                   524                                 50    2020-05-30 02:30:00 CST                 4.360360e+05                          0.0  Agent_ping(30), Disk_rd_ios(30), Disk_io_util(...```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse metric_df if present, else load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# 1) Global P95 and P5 per (cmdb_id, kpi_name)
thresholds = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(global_P95=lambda x: x.quantile(0.95),
         global_P5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) Filter to incident window
start_ts = 1590777000
end_ts   = 1590778800
window_df = metric_df[(metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)].copy()

# Merge thresholds into window
w = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

# If no anomalies at all, exit with message
if w['is_anomaly'].sum() == 0:
    result = "No faults (consecutive anomalous runs) found in the incident window based on global P95/P5 thresholds."
else:
    # Recompute per-cmdb totals to pick top10 later
    per_cmdb_totals = w.groupby('cmdb_id')['is_anomaly'].sum().reset_index(name='total_anomaly_points')
    top10_cmdbs = per_cmdb_totals.sort_values('total_anomaly_points', ascending=False).head(10)['cmdb_id'].tolist()

    # Work on anomaly rows only and group by (cmdb_id,kpi_name)
    anom = w[w['is_anomaly']].sort_values(['cmdb_id','kpi_name','timestamp']).copy()

    faults = []
    eps = 1e-9

    # Iterate groups and find consecutive runs (diff == 60)
    for (cmdb, kpi), g in anom.groupby(['cmdb_id','kpi_name'], sort=False):
        ts = g['timestamp'].to_numpy(dtype='int64')
        vals = g['value'].to_numpy(dtype='float64')
        gp95 = float(g['global_P95'].iloc[0])
        gp5  = float(g['global_P5'].iloc[0])

        if len(ts) == 0:
            continue

        # identify breaks where diff != 60
        diffs = np.diff(ts, prepend=ts[0])  # first diff 0
        # start new fault when diff != 60 (for first element diffs[0]==0 => new fault)
        fault_ids = (diffs != 60).cumsum()

        for fid, idxs in g.groupby(fault_ids).groups.items():
            seg = g.loc[idxs]
            start_epoch = int(seg['timestamp'].min())
            end_epoch = int(seg['timestamp'].max())
            length_minutes = int((end_epoch - start_epoch) / 60) + 1
            max_val = float(seg['value'].max())
            min_val = float(seg['value'].min())

            high_flag = max_val >= gp95
            low_flag  = min_val <= gp5

            high_breach_ratio = (max_val - gp95) / max(eps, abs(gp95))
            low_breach_ratio  = (gp5 - min_val) / max(eps, abs(gp5))

            # choose anomaly_type per instructions
            if high_flag and not low_flag:
                anomaly_type = 'high'
                breach_ratio = high_breach_ratio
            elif low_flag and not high_flag:
                anomaly_type = 'low'
                breach_ratio = low_breach_ratio
            elif high_flag and low_flag:
                if high_breach_ratio >= low_breach_ratio:
                    anomaly_type = 'high'
                    breach_ratio = high_breach_ratio
                else:
                    anomaly_type = 'low'
                    breach_ratio = low_breach_ratio
            else:
                # shouldn't happen, but skip if no flag
                continue

            faults.append({
                'cmdb_id': cmdb,
                'kpi_name': kpi,
                'fault_start_epoch': start_epoch,
                'fault_end_epoch': end_epoch,
                'fault_length_minutes': length_minutes,
                'anomaly_type': anomaly_type,
                'global_P95': gp95,
                'global_P5': gp5,
                'max_value_in_fault': max_val,
                'min_value_in_fault': min_val,
                'breach_ratio': breach_ratio
            })

    if len(faults) == 0:
        result = "No faults found in the incident window after grouping consecutive anomalies."
    else:
        faults_df = pd.DataFrame(faults)

        # 6) Keep only faults for top10 cmdb_ids by total_anomaly_points
        faults_top = faults_df[faults_df['cmdb_id'].isin(top10_cmdbs)].copy()
        if faults_top.shape[0] == 0:
            result = "No faults found for top 10 cmdb_ids by anomaly volume in the incident window."
        else:
            # Sort by fault_length_minutes desc then breach_ratio desc
            faults_top = faults_top.sort_values(['fault_length_minutes','breach_ratio'], ascending=[False, False]).reset_index(drop=True)

            # Format timestamps to UTC+8 strings
            faults_top['fault_start_timestamp'] = pd.to_datetime(faults_top['fault_start_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')
            faults_top['fault_end_timestamp']   = pd.to_datetime(faults_top['fault_end_epoch'], unit='s', utc=True).dt.tz_convert(tz).dt.strftime('%Y-%m-%d %H:%M:%S %Z')

            # Select and round columns
            out_cols = [
                'cmdb_id','kpi_name','fault_start_timestamp','fault_end_timestamp',
                'fault_length_minutes','anomaly_type','global_P95','global_P5',
                'max_value_in_fault','min_value_in_fault','breach_ratio'
            ]
            faults_top[out_cols[6:]] = faults_top[out_cols[6:]].astype(float).round(6)
            result = faults_top[out_cols].head(20).reset_index(drop=True)

# Display result (DataFrame or message)
result

```
Out[4]:
```
```python
# Plain-English summary of the detected faults (compact)
summary = (
    "Faults summary (incident window 2020-05-30 02:30–03:00 UTC+8):\n"
    "- Faults are widespread and start at the window boundary (earliest faults begin at 2020-05-30 02:30:00 UTC+8).\n"
    "- Top faults (examples from the top 20):\n"
    "  * os_007 / Disk_rd_ios: 31-minute fault (02:30–03:00), type=high, values 0 (global P95/P5 both 0).\n"
    "  * os_006 / Disk_svctm: 30-minute fault (02:30–02:59), type=high, breach_ratio extremely large (global P95=0, max=1.5).\n"
    "  * db_002 / Login_Per_Sec: 30-minute fault (02:30–02:59), type=high, breach_ratio ≈ 4.75 (max 2.3 vs P95 0.4).\n"
    "  * db_005 / AIOS: 30-minute fault, type=high, breach_ratio = 1.0 (max 0.02 vs P95 0.01).\n"
    "  * db_011 / ACS: 30-minute fault, type=low, breach_ratio = 1.0 (min 0.0 vs P5 0.01).\n"
    "- Most faults last the entire observed span for that KPI in-window (30 or 31 minutes) and many metrics are constant during the fault (max == min).\n"
    "- Several global thresholds (P95 or P5) are zero or equal to observed values; that yields zero or extremely large breach_ratio values and indicates boundary conditions in thresholding.\n"
    "- Overall interpretation: many hosts/services (OS and DB nodes) show sustained anomalous metric values beginning at 02:30. This pattern suggests a systemic issue at that time rather than isolated short spikes. Recommended next steps: investigate the listed hosts (e.g., os_007, os_006, db_002, db_005, db_011, db_013) for common causes (resource exhaustion, I/O subsystem, or network/service-wide problem) and examine logs around 02:30 for correlated events."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id               kpi_name    fault_start_timestamp      fault_end_timestamp  fault_length_minutes anomaly_type  global_P95   global_P5  max_value_in_fault  min_value_in_fault  breach_ratio
0   os_007            Disk_rd_ios  2020-05-30 02:30:00 CST  2020-05-30 03:00:00 CST                    31         high    0.000000    0.000000            0.000000            0.000000  0.000000e+00
1   os_006             Disk_svctm  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.000000    0.000000            1.500000            0.000000  1.500000e+09
2   db_002          Login_Per_Sec  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.400000    0.300000            2.300000            0.300000  4.750000e+00
3   db_005                   AIOS  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.010000    0.000000            0.020000            0.000000  1.000000e+00
4   db_011                    ACS  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30          low    0.010500    0.010000            0.020000            0.000000  1.000000e+00
5   db_013          Login_Per_Sec  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.200000    0.100000            0.400000            0.100000  1.000000e+00
6   os_007             Disk_await  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    1.025000    0.000000            1.660000            0.000000  6.195120e-01
7   db_002                    ACS  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.020000    0.010000            0.030000            0.020000  5.000000e-01
8   db_008            Sess_Active  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.400000    0.100000            0.500000            0.100000  2.500000e-01
9   os_007  Processor_load_15_min  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.008333    0.008333            0.010000            0.008333  2.000480e-01
10  os_006  Processor_load_15_min  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.008416    0.008333            0.010000            0.008333  1.881640e-01
11  db_008            TPS_Per_Sec  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.700000    0.600000            0.800000            0.600000  1.428570e-01
12  db_010  Physical_Read_Per_Sec  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high  117.800000    0.000000          119.800000            0.000000  1.697800e-02
13  os_006        Memory_used_pct  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30          low   16.770000   16.769500           16.770000           16.760000  5.670000e-04
14  db_002                   AIOS  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    0.000000    0.000000            0.000000            0.000000  0.000000e+00
15  db_002                   Hang  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    1.000000    1.000000            1.000000            1.000000  0.000000e+00
16  db_002              MEM_Total  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high  381.902264  381.902264          381.902264          381.902264  0.000000e+00
17  db_002           On_Off_State  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    1.000000    1.000000            1.000000            1.000000  0.000000e+00
18  db_002          Proc_Used_Pct  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    1.370000    1.360000            1.370000            1.360000  0.000000e+00
19  db_002     Proc_User_Used_Pct  2020-05-30 02:30:00 CST  2020-05-30 02:59:00 CST                    30         high    1.370000    1.360000            1.370000            1.360000  0.000000e+00```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz

# reuse trace_df if available, else load
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')

trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

tz = pytz.timezone('Asia/Shanghai')

# 1) global P95 and P5 per (cmdb_id, trace_name)
trace_thresholds = (
    trace_df
    .groupby(['cmdb_id','trace_name'])['value']
    .agg(global_P95=lambda x: x.quantile(0.95),
         global_P5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# 2) filter to incident window inclusive
start_ts = 1590777000
end_ts   = 1590778800
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

if trace_window.shape[0] == 0:
    result = "No trace data found in the incident window 1590777000..1590778800."
else:
    # 3) merge thresholds and flag anomalies
    tw = trace_window.merge(trace_thresholds, on=['cmdb_id','trace_name'], how='left')
    tw['is_anomaly'] = ((tw['value'] >= tw['global_P95']) | (tw['value'] <= tw['global_P5'])).fillna(False)

    # 4) compute aggregates
    anomaly_counts = (
        tw
        .groupby(['cmdb_id','trace_name'], as_index=False)['is_anomaly']
        .sum()
        .rename(columns={'is_anomaly':'anomaly_count_in_window'})
    )

    earliest = (
        tw[tw['is_anomaly']]
        .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_epoch'})
    )

    maxmin = (
        tw
        .groupby(['cmdb_id','trace_name'], as_index=False)['value']
        .agg(max_value_in_window='max', min_value_in_window='min')
    )

    # merge pieces with thresholds
    agg = (
        trace_thresholds
        .merge(anomaly_counts, on=['cmdb_id','trace_name'], how='left')
        .merge(earliest, on=['cmdb_id','trace_name'], how='left')
        .merge(maxmin, on=['cmdb_id','trace_name'], how='left')
    )

    # format earliest timestamp to UTC+8 string
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_epoch'].apply(fmt_epoch)

    # round numeric columns for compactness
    for col in ['global_P95','global_P5','max_value_in_window','min_value_in_window']:
        if col in agg.columns:
            agg[col] = agg[col].astype(float).round(6)

    # keep only rows with anomalies (>0)
    agg_nonzero = agg[agg['anomaly_count_in_window'].fillna(0) > 0].copy()

    if agg_nonzero.shape[0] == 0:
        result = "No anomalies detected in trace data within the incident window based on global P95/P5 thresholds."
    else:
        result = (
            agg_nonzero
            .loc[:, ['cmdb_id','trace_name','global_P95','global_P5','anomaly_count_in_window',
                     'earliest_anomaly_timestamp','max_value_in_window','min_value_in_window']]
            .sort_values('anomaly_count_in_window', ascending=False)
            .head(20)
            .reset_index(drop=True)
        )

# final output (DataFrame or message)
result

```
Out[5]:
```
```python
# Plain-English summary of trace-level anomaly scan (compact)
try:
    df = result  # reuse result from previous trace analysis
except NameError:
    df = None

if df is None or isinstance(df, str):
    summary = "No trace anomaly results available to summarize."
else:
    # Top candidates by anomaly count
    top_rows = df.head(5)
    top_list = [f"{r.cmdb_id}/{r.trace_name}({int(r.anomaly_count_in_window)})" for r in top_rows.itertuples(index=False)]
    shown = int(df.shape[0])
    total_anomalies = int(df['anomaly_count_in_window'].sum())
    earliest_examples = df['earliest_anomaly_timestamp'].dropna().unique()[:3].tolist()
    # Observations about values and thresholds
    zero_threshold_count = int(((df['global_P95'].astype(float) == 0) & (df['global_P5'].astype(float) == 0)).sum())
    zero_value_count = int(((df['max_value_in_window'].astype(float) == 0) & (df['min_value_in_window'].astype(float) == 0)).sum())

    summary = (
        "Trace anomaly scan summary (incident window 2020-05-30 02:30–03:00 UTC+8):\n"
        f"- Top candidates (examples): {', '.join(top_list)}.\n"
        f"- Showing {shown} trace rows (top 20). Total anomaly points across these rows: {total_anomalies}.\n"
        f"- Earliest anomaly timestamps (examples): {', '.join(earliest_examples)}.\n"
        f"- {zero_threshold_count} of the shown trace metrics have both global P95 and P5 equal to 0.\n"
        f"- {zero_value_count} of the shown trace metrics have all in-window values equal to 0 (max==min==0).\n"
        "- Interpretation: Many trace error-rate metrics are flagged as anomalous throughout the window,\n"
        "  often because global thresholds and observed values are at or near zero (yielding repeated boundary matches).\n"
        "- Actionable next steps: inspect the top affected hosts (e.g., docker_001, docker_002, os_021/os_022) and trace flows between them\n"
        "  around 02:30 for root causes (instrumentation issues, pervasive zero/error-rate reporting, or systemic failures)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                      trace_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0   docker_001      trace.from_os_022.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
1   docker_002      trace.from_os_022.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
2   docker_003      trace.from_os_021.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
3       os_022    trace.to_docker_001.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
4       os_021    trace.to_docker_003.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
5       os_021    trace.to_docker_004.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
6       os_021                       trace.self.csf.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
7       os_021                  trace.from_root.osb.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
8       os_022                  trace.from_root.osb.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
9       os_022                       trace.self.csf.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
10      os_022    trace.to_docker_002.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
11  docker_004      trace.from_os_021.remoteprocess.error_rate         0.0        0.0                       30    2020-05-30 02:30:00 CST                  0.0                  0.0
12  docker_001                     trace.self.local.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
13  docker_001    trace.to_docker_008.remoteprocess.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
14  docker_001                      trace.self.jdbc.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
15  docker_001                       trace.self.csf.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
16  docker_007  trace.from_docker_001.remoteprocess.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
17  docker_007                     trace.self.local.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
18  docker_005                     trace.self.local.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0
19  docker_006                      trace.self.jdbc.error_rate         0.0        0.0                       29    2020-05-30 02:30:00 CST                  0.0                  0.0```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz

# reuse trace_df and tz if available
try:
    trace_df
except NameError:
    trace_df = pd.read_csv('trace.csv')
    trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# 1) Select trace_name that contain 'duration_mean' or 'duration_p95' (case-sensitive as in data)
mask_duration = trace_df['trace_name'].str.contains('duration_mean|duration_p95', na=False)
duration_series = trace_df[mask_duration].copy()

# If no duration series present, return message
if duration_series.shape[0] == 0:
    result = "No duration anomalies found in window."
else:
    # Compute global P95 per (cmdb_id, trace_name) using full series (rule: before filtering window)
    duration_thresholds = (
        duration_series
        .groupby(['cmdb_id','trace_name'])['value']
        .agg(global_P95=lambda x: x.quantile(0.95))
        .reset_index()
    )

    # 2) Filter to incident window inclusive
    start_ts = 1590777000
    end_ts = 1590778800
    duration_window = duration_series[(duration_series['timestamp'] >= start_ts) & (duration_series['timestamp'] <= end_ts)].copy()

    if duration_window.shape[0] == 0:
        result = "No duration anomalies found in window."
    else:
        # 3) Merge thresholds and flag anomalies where value >= global_P95
        dw = duration_window.merge(duration_thresholds, on=['cmdb_id','trace_name'], how='left')
        dw['is_anomaly'] = (dw['value'] >= dw['global_P95']).fillna(False)

        # 4) Compute aggregates per (cmdb_id, trace_name)
        agg_counts = (
            dw
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(
                global_P95=('global_P95','first'),
                anomaly_count_in_window=('is_anomaly','sum'),
                earliest_anomaly_epoch=('timestamp', lambda s: s[dw.loc[s.index,'is_anomaly']].min() if s.shape[0]>0 else pd.NA),
                max_value_in_window=('value','max'),
                min_value_in_window=('value','min')
            )
        )

        # The lambda above can produce incorrect result due to groupby index mismatch; safer compute earliest separately:
        earliest = (
            dw[dw['is_anomaly']]
            .groupby(['cmdb_id','trace_name'], as_index=False)['timestamp']
            .min()
            .rename(columns={'timestamp':'earliest_anomaly_epoch'})
        )

        agg_simple = (
            dw
            .groupby(['cmdb_id','trace_name'], as_index=False)
            .agg(global_P95=('global_P95','first'),
                 anomaly_count_in_window=('is_anomaly','sum'),
                 max_value_in_window=('value','max'),
                 min_value_in_window=('value','min'))
        )

        agg_merged = agg_simple.merge(earliest, on=['cmdb_id','trace_name'], how='left')

        # Format earliest timestamp to UTC+8
        def fmt_epoch(e):
            if pd.isna(e):
                return pd.NA
            return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

        agg_merged['earliest_anomaly_timestamp'] = agg_merged['earliest_anomaly_epoch'].apply(fmt_epoch)

        # Round numeric columns for compactness
        agg_merged['global_P95'] = agg_merged['global_P95'].astype(float).round(6)
        agg_merged['max_value_in_window'] = agg_merged['max_value_in_window'].astype(float).round(6)
        agg_merged['min_value_in_window'] = agg_merged['min_value_in_window'].astype(float).round(6)

        # Keep only rows with anomaly_count_in_window > 0
        agg_nonzero = agg_merged[agg_merged['anomaly_count_in_window'] > 0].copy()

        if agg_nonzero.shape[0] == 0:
            result = "No duration anomalies found in window."
        else:
            # 5) Sort by anomaly_count_in_window desc and limit to top 20
            result = (
                agg_nonzero
                .loc[:, ['cmdb_id','trace_name','global_P95','anomaly_count_in_window',
                         'earliest_anomaly_timestamp','max_value_in_window','min_value_in_window']]
                .sort_values('anomaly_count_in_window', ascending=False)
                .head(20)
                .reset_index(drop=True)
            )

# Display the result (DataFrame or message)
result

```
Out[6]:
```
```python
# Summarize duration anomaly scan results in plain English (compact)
try:
    df = result  # reuse result DataFrame from previous step
except NameError:
    df = None

if df is None or isinstance(df, str):
    summary = "No duration anomalies found in window."
else:
    n_rows = int(df.shape[0])
    total_anomaly_points = int(df['anomaly_count_in_window'].sum())
    earliest_examples = df['earliest_anomaly_timestamp'].dropna().unique()[:3].tolist()
    # Top 5 by anomaly count with P95 and max values
    top5 = []
    for r in df.head(5).itertuples(index=False):
        top5.append(f"{r.cmdb_id}/{r.trace_name} count={int(r.anomaly_count_in_window)} P95={float(r.global_P95):.6g} max={float(r.max_value_in_window):.6g}")
    # Highlight traces with large global P95 (>= 1s)
    large_p95 = df[df['global_P95'].astype(float) >= 1.0]
    large_p95_examples = []
    for r in large_p95.head(5).itertuples(index=False):
        large_p95_examples.append(f"{r.cmdb_id}/{r.trace_name} P95={float(r.global_P95):.6g} max={float(r.max_value_in_window):.6g}")

    summary = (
        "Duration anomaly scan (window 2020-05-30 02:30–03:00 UTC+8):\n"
        f"- Returned rows: {n_rows} duration-series candidates with anomalies. Total anomaly points (sum of counts): {total_anomaly_points}.\n"
        f"- Earliest anomalies start at: {', '.join(earliest_examples)} (examples).\n"
        f"- Top candidates (examples):\n    " + "\n    ".join(top5) + "\n"
    )

    if large_p95_examples:
        summary += (
            "- Notable high-latency traces (global P95 >= 1s) include:\n    "
            + "\n    ".join(large_p95_examples) + "\n"
        )

    summary += (
        "- Interpretation: several JDBC and remote-process duration p95 metrics (e.g., docker_003 trace.self.jdbc.duration_p95 with 27 anomaly points)\n"
        "  exceeded their historical P95 repeatedly during the incident window, and some remote/flyremote traces show very large P95 values (~20s),\n"
        "  indicating elevated latency on those call paths. Investigate the top affected containers and remote calls (docker_003, docker_004, docker_001, os_021, etc.)\n"
        "  around 02:30 for root causes (DB latency, network delays, or downstream service slowness)."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                        trace_name  global_P95  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window
0   docker_003                      trace.self.jdbc.duration_p95    0.003000                       27    2020-05-30 02:30:00 CST             0.003250             0.002250
1   docker_004                      trace.self.jdbc.duration_p95    0.003000                       26    2020-05-30 02:30:00 CST             0.003000             0.002250
2   docker_007                     trace.self.local.duration_p95    0.010000                        8    2020-05-30 02:30:00 CST             0.018000             0.006850
3   docker_006                     trace.self.local.duration_p95    0.011000                        4    2020-05-30 02:33:00 CST             0.022000             0.007000
4   docker_004                 trace.self.flyremote.duration_p95   20.001000                        4    2020-05-30 02:44:00 CST            20.001650             0.007000
5   docker_001    trace.to_docker_007.remoteprocess.duration_p95    0.037100                        3    2020-05-30 02:40:00 CST             0.054150             0.022000
6   docker_003      trace.from_os_021.remoteprocess.duration_p95   20.009000                        3    2020-05-30 02:44:00 CST            20.012400             0.714700
7   docker_007  trace.from_docker_001.remoteprocess.duration_p95    0.037100                        3    2020-05-30 02:40:00 CST             0.054150             0.022000
8       os_021    trace.to_docker_003.remoteprocess.duration_p95   20.009000                        3    2020-05-30 02:44:00 CST            20.012400             0.714700
9   docker_001                     trace.self.jdbc.duration_mean    0.004335                        2    2020-05-30 02:31:00 CST             0.004763             0.000875
10  docker_001                      trace.self.jdbc.duration_p95    0.007600                        2    2020-05-30 02:31:00 CST             0.026900             0.002000
11  docker_001                    trace.self.local.duration_mean    0.023643                        2    2020-05-30 02:52:00 CST             0.028550             0.002071
12  docker_001   trace.to_docker_008.remoteprocess.duration_mean    0.026987                        2    2020-05-30 02:31:00 CST             0.028760             0.022333
13  docker_001    trace.to_docker_008.remoteprocess.duration_p95    0.039480                        2    2020-05-30 02:40:00 CST             0.058000             0.024800
14  docker_001                     trace.self.local.duration_p95    0.099600                        2    2020-05-30 02:40:00 CST             0.101000             0.004100
15  docker_001   trace.to_docker_007.remoteprocess.duration_mean    0.026749                        2    2020-05-30 02:40:00 CST             0.028971             0.022000
16  docker_002                      trace.self.csf.duration_mean    0.075209                        2    2020-05-30 02:41:00 CST             0.076331             0.028500
17  docker_002                       trace.self.csf.duration_p95    0.195650                        2    2020-05-30 02:37:00 CST             0.197000             0.036500
18  docker_001                 trace.self.flyremote.duration_p95   20.012820                        2    2020-05-30 02:46:00 CST            20.061000             0.007000
19  docker_002                 trace.self.flyremote.duration_p95   20.025732                        2    2020-05-30 02:44:00 CST            20.061000             0.007950```
```

In [ ]:
# In[7]:

import pandas as pd
import pytz

# Reuse metric_df and tz if present; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Focus on cmdb_id = 'db_006'
target_cmdb = 'db_006'

# 1) Identify all kpi_name for db_006 in the full metric.csv
db6_all_kpis = metric_df[metric_df['cmdb_id'] == target_cmdb]['kpi_name'].dropna().unique()
db6_all_kpis = sorted(db6_all_kpis)

# 2) Compute global P95 and P5 for each (db_006, kpi_name) using full series (before window filtering)
db6_thresholds = (
    metric_df[metric_df['cmdb_id'] == target_cmdb]
    .groupby(['cmdb_id','kpi_name'])['value']
    .agg(global_P95=lambda x: x.quantile(0.95),
         global_P5 =lambda x: x.quantile(0.05))
    .reset_index()
)

# 3) Filter to incident window (inclusive)
start_ts = 1590777000
end_ts   = 1590778800
db6_window = metric_df[(metric_df['cmdb_id'] == target_cmdb) &
                       (metric_df['timestamp'] >= start_ts) &
                       (metric_df['timestamp'] <= end_ts)].copy()

# If no data in window for db_006, return explicit message
if db6_window.shape[0] == 0:
    result = f"No metric data for {target_cmdb} in the incident window 1590777000..1590778800."
else:
    # 4) Merge thresholds and flag anomalies in-window (value >= global_P95 OR value <= global_P5)
    w = db6_window.merge(db6_thresholds, on=['cmdb_id','kpi_name'], how='left')
    w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

    # Aggregate per kpi in-window
    agg = (
        w.groupby('kpi_name', as_index=False)
         .agg(
             global_P95=('global_P95','first'),
             global_P5 =('global_P5','first'),
             anomaly_count_in_window=('is_anomaly','sum'),
             earliest_anomaly_epoch=('timestamp', lambda s: s[w.loc[s.index,'is_anomaly']].min() if s.shape[0]>0 else pd.NA),
             max_value_in_window=('value','max'),
             min_value_in_window=('value','min'),
             mean_value_in_window=('value','mean')
         )
    )

    # The lambda above can be fragile due to index alignment; compute earliest safely:
    earliest = (
        w[w['is_anomaly']]
        .groupby('kpi_name', as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_epoch'})
    )

    agg_simple = (
        w.groupby('kpi_name', as_index=False)
         .agg(global_P95=('global_P95','first'),
              global_P5 =('global_P5','first'),
              anomaly_count_in_window=('is_anomaly','sum'),
              max_value_in_window=('value','max'),
              min_value_in_window=('value','min'),
              mean_value_in_window=('value','mean'))
    )

    agg_merged = agg_simple.merge(earliest, on='kpi_name', how='left')

    # Format earliest timestamp to UTC+8 string
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    agg_merged['earliest_anomaly_timestamp'] = agg_merged['earliest_anomaly_epoch'].apply(fmt_epoch)

    # Round numeric columns for compactness
    for col in ['global_P95','global_P5','max_value_in_window','min_value_in_window','mean_value_in_window']:
        if col in agg_merged.columns:
            agg_merged[col] = agg_merged[col].astype(float).round(6)

    # Keep only KPIs that appear in the window (they are already) and sort by anomaly_count desc
    result_df = agg_merged.loc[:, [
        'kpi_name','global_P95','global_P5','anomaly_count_in_window',
        'earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','mean_value_in_window'
    ]].sort_values('anomaly_count_in_window', ascending=False).reset_index(drop=True)

    # If no anomalies found for db_006, return explicit message
    if result_df['anomaly_count_in_window'].sum() == 0:
        result = f"No anomalies found for {target_cmdb} in the incident window based on global P95/P5 thresholds."
    else:
        result = result_df.head(20)

# Display result (DataFrame or message)
result

```
Out[7]:
```
- Scope: focused on db_006 metrics during the incident window 2020-05-30 02:30:00–03:00:00 (UTC+8).

- KPIs observed: db_006 has many KPIs; top ones by anomaly count (shown up to 20) include AIOS, Hang, TPS_Per_Sec, Row_Lock, SctRead_Per_Sec, Proc_Used_Pct, Proc_User_Used_Pct, On_Off_State, MEM_real_util, MEM_Total, Login_Per_Sec, User_Commit, Sess_Active, tnsping_result_time, AWS, Physical_Read_Per_Sec, LFSync_Per_Sec, Session_pct, DFParaWrite_Per_Sec, LFParaWrite_Per_Sec.

- Global thresholds (P95/P5): computed from the full historical series per (db_006, kpi). Examples:
  - AIOS: P95=0.01, P5=0.00
  - Hang: P95=P5=1.00
  - TPS_Per_Sec: P95=1.2, P5=1.0
  - MEM_Total: P95=P5=508.25241
  - Physical_Read_Per_Sec: P95=71.305, P5=0.0
  (full per-KPI values are in the returned table.)

- Anomalies in the incident window:
  - Many KPIs show anomaly_count_in_window = 30 (i.e., flagged every minute across the 30-minute window). Examples: AIOS, Hang, TPS_Per_Sec, Row_Lock, SctRead_Per_Sec, Proc_Used_Pct, Proc_User_Used_Pct, On_Off_State, MEM_real_util, MEM_Total, Login_Per_Sec, User_Commit, Sess_Active, tnsping_result_time.
  - Some KPIs have slightly fewer anomaly minutes (e.g., AWS 28, Physical_Read_Per_Sec 27, LFSync_Per_Sec 26, Session_pct 25, DFParaWrite 24, LFParaWrite 21).

- Timing: the earliest anomaly for the majority of KPIs is 2020-05-30 02:30:00 (UTC+8). A few KPIs first breached a bit later (e.g., LFParaWrite_Per_Sec at 02:31, LFSync_Per_Sec at 02:32).

- Values behavior:
  - Many KPIs are constant during the window (max == min), e.g., Hang (max=min=1.0), MEM_Total constant at 508.25241, On_Off_State constant 1.0 — these hit their P95/P5 boundary every minute.
  - Some KPIs show meaningful variation & elevated peaks vs. historical P95, e.g., Physical_Read_Per_Sec (max 72.3, P95 71.305), DFParaWrite_Per_Sec and LFParaWrite_Per_Sec show non-zero maxima above historical thresholds.

- Interpretation:
  - db_006 exhibits pervasive, sustained anomalies across many KPIs starting at the window start (02:30). This pattern suggests a systemic condition affecting the node/service rather than isolated transient spikes.
  - The presence of constant boundary values (P95==value or P5==value) for many KPIs indicates either sustained abnormal state or thresholds at boundary values that cause repeated matches.

- Suggested next actions:
  - Investigate db_006 logs and system events around 02:30 for configuration changes, restarts, or resource issues.
  - Check I/O and DB activity (Physical_Read_Per_Sec, DF/LF writes, TPS/Login metrics) to determine whether DB load or I/O subsystem problems caused sustained anomalies.
  - Correlate with host-level metrics (CPU, disk I/O, network) and with other top affected nodes to identify common root causes.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0                    AIOS     0.01000    0.00000                       30    2020-05-30 02:30:00 CST              0.01000              0.00000              0.000333
1                    Hang     1.00000    1.00000                       30    2020-05-30 02:30:00 CST              1.00000              1.00000              1.000000
2             TPS_Per_Sec     1.20000    1.00000                       30    2020-05-30 02:30:00 CST              1.20000              1.00000              1.013333
3                Row_Lock     0.00000    0.00000                       30    2020-05-30 02:30:00 CST              0.00000              0.00000              0.000000
4         SctRead_Per_Sec     0.00000    0.00000                       30    2020-05-30 02:30:00 CST              0.00000              0.00000              0.000000
5           Proc_Used_Pct     2.66000    2.65000                       30    2020-05-30 02:30:00 CST              2.65000              2.65000              2.650000
6      Proc_User_Used_Pct     2.66000    2.65000                       30    2020-05-30 02:30:00 CST              2.65000              2.65000              2.650000
7            On_Off_State     1.00000    1.00000                       30    2020-05-30 02:30:00 CST              1.00000              1.00000              1.000000
8           MEM_real_util     0.00000    0.00000                       30    2020-05-30 02:30:00 CST              0.00000              0.00000              0.000000
9               MEM_Total   508.25241  508.25241                       30    2020-05-30 02:30:00 CST            508.25241            508.25241            508.252410
10          Login_Per_Sec     0.10000    0.00000                       30    2020-05-30 02:30:00 CST              0.10000              0.00000              0.053333
11            User_Commit     0.18000    0.00000                       30    2020-05-30 02:30:00 CST              0.18000              0.00000              0.012000
12            Sess_Active     0.20000    0.10000                       30    2020-05-30 02:30:00 CST              0.20000              0.10000              0.110000
13    tnsping_result_time    10.00000    0.00000                       30    2020-05-30 02:30:00 CST             10.00000              0.00000              1.333333
14                    AWS     0.02000    0.00000                       28    2020-05-30 02:30:00 CST              0.02000              0.00000              0.004000
15  Physical_Read_Per_Sec    71.30500    0.00000                       27    2020-05-30 02:30:00 CST             72.30000              0.00000              7.253333
16         LFSync_Per_Sec     2.51500    0.00000                       26    2020-05-30 02:32:00 CST              2.80000              0.00000              0.306000
17            Session_pct     0.04200    0.01500                       25    2020-05-30 02:30:00 CST              0.05400              0.01500              0.018700
18    DFParaWrite_Per_Sec     1.39900    0.00000                       24    2020-05-30 02:30:00 CST              1.83000              0.00000              0.301667
19    LFParaWrite_Per_Sec     1.71600    0.00000                       21    2020-05-30 02:31:00 CST              1.86000              0.00000              0.554667```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import re

# Reuse metric_df and tz from session if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Parameters
target_db = 'db_006'
start_ts = 1590777000
end_ts = 1590778800

# 1) Try to identify corresponding OS host for db_006 by numeric suffix match (e.g., os_006)
m = re.search(r'_(\d+)$', target_db)
os_candidate = None
if m:
    suffix = m.group(1)
    candidate = f'os_{suffix}'
    if candidate in metric_df['cmdb_id'].unique():
        os_candidate = candidate

# If no direct suffix match, try to find any os_* that exist and choose the most likely by name similarity:
if os_candidate is None:
    os_list = [c for c in metric_df['cmdb_id'].unique() if isinstance(c, str) and c.startswith('os_')]
    if len(os_list) == 1:
        os_candidate = os_list[0]
    else:
        # no deterministic mapping found
        os_candidate = None

if os_candidate is None:
    result = f"No OS host mapping found for {target_db} in metric.csv (cannot determine os_*)."
else:
    # 2) List disk-related KPI names for that OS host
    os_df = metric_df[metric_df['cmdb_id'] == os_candidate].copy()
    # disk KPI patterns (common names)
    disk_patterns = ['Disk_io_util','Disk_rd_ios','Disk_wr_ios','Disk_rd_kbs','Disk_wr_kbs',
                     'Disk_svctm','Disk_await','Disk_avgqu_sz','Disk_rd_kbs','Disk_wr_kbs','Disk_rd_ios','Disk_wr_ios']
    # also include any kpi_name that contains 'Disk' (case-sensitive as in data)
    disk_kpis = sorted(os_df[os_df['kpi_name'].str.contains('Disk', na=False)]['kpi_name'].unique().tolist())

    if len(disk_kpis) == 0:
        result = f"OS host {os_candidate} found but no Disk-* KPIs present in metric.csv."
    else:
        # 3) Compute global P95 and P5 per (os_host, kpi_name) using full series (before window filtering)
        thresholds = (
            metric_df[metric_df['cmdb_id'] == os_candidate]
            .groupby(['cmdb_id','kpi_name'])['value']
            .agg(global_P95=lambda x: x.quantile(0.95),
                 global_P5 =lambda x: x.quantile(0.05))
            .reset_index()
        )

        # Keep only disk KPIs of interest
        thresholds_disk = thresholds[thresholds['kpi_name'].isin(disk_kpis)].copy()

        # 4) Filter metric.csv to incident window for this OS host
        window_os = metric_df[
            (metric_df['cmdb_id'] == os_candidate) &
            (metric_df['timestamp'] >= start_ts) &
            (metric_df['timestamp'] <= end_ts)
        ].copy()

        if window_os.shape[0] == 0:
            result = f"No metric data for {os_candidate} in the incident window 1590777000..1590778800."
        else:
            # 5) Merge thresholds and flag anomalies in-window (value >= P95 OR value <= P5)
            w = window_os.merge(thresholds_disk, on=['cmdb_id','kpi_name'], how='left')
            w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

            # Aggregate per kpi_name
            # anomaly_count, earliest anomaly epoch, max/min/mean in-window
            anomaly_counts = w.groupby('kpi_name', as_index=False)['is_anomaly'].sum().rename(columns={'is_anomaly':'anomaly_count_in_window'})
            earliest = (w[w['is_anomaly']].groupby('kpi_name', as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_epoch'}))
            stats = w.groupby('kpi_name', as_index=False)['value'].agg(max_value_in_window='max', min_value_in_window='min', mean_value_in_window='mean')

            merged = thresholds_disk.merge(anomaly_counts, on='kpi_name', how='right') \
                                     .merge(earliest, on='kpi_name', how='left') \
                                     .merge(stats, on='kpi_name', how='left')

            # Format earliest timestamp to UTC+8
            def fmt_epoch(e):
                if pd.isna(e):
                    return pd.NA
                return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

            merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_epoch'].apply(fmt_epoch)

            # Round numeric columns for compactness
            for col in ['global_P95','global_P5','max_value_in_window','min_value_in_window','mean_value_in_window']:
                if col in merged.columns:
                    merged[col] = merged[col].astype(float).round(6)

            # 6) Prepare final compact table sorted by anomaly_count desc, include cmdb_id column
            merged['cmdb_id'] = os_candidate
            out_cols = ['cmdb_id','kpi_name','global_P95','global_P5','anomaly_count_in_window',
                        'earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','mean_value_in_window']
            final = merged[out_cols].sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

            # If no anomalies for disk KPIs, explicit message
            if final['anomaly_count_in_window'].fillna(0).sum() == 0:
                result = f"No disk KPI anomalies found for {os_candidate} in the incident window based on global P95/P5 thresholds."
            else:
                result = final

# Display result variable (DataFrame or message)
result

```
Out[8]:
```
- OS host mapping: db_006 maps to os_006 (by numeric suffix). os_006 metrics were examined.

- Disk-related KPIs found for os_006 (examples present in metric.csv): Disk_rd_kbs, Disk_svctm, Disk_io_util, Disk_wr_ios, Disk_rd_ios, Disk_await, Disk_avgqu_sz, Disk_wr_kbs (plus other non-disk OS KPIs listed).

- Global thresholds (P95/P5) and in-window behavior (incident window 2020-05-30 02:30–03:00 UTC+8):
  - Disk_rd_kbs: global P95=0, P5=0 → anomaly_count_in_window=30, earliest=02:30, max=0, min=0, mean=0
  - Disk_svctm: global P95=0, P5=0 → anomaly_count_in_window=30, earliest=02:30, max=1.5, min=0, mean≈0.05
  - Disk_io_util: global P95=0, P5=0 → anomaly_count_in_window=30, earliest=02:30, max=0, min=0, mean=0
  - Disk_wr_ios: global P95=0, P5=0 → anomaly_count_in_window=30, earliest=02:30, max=0, min=0, mean=0
  - Disk_rd_ios: global P95=0, P5=0 → anomaly_count_in_window=30, earliest=02:30, max=0, min=0, mean=0
  - Disk_await: global P95≈1.3385, P5=0 → anomaly_count_in_window=20, earliest=02:30, max=2.5, min=0, mean≈0.528
  - Disk_avgqu_sz: global P95≈0.00166, P5≈0.000497 → anomaly_count_in_window=3, earliest=02:36, max≈0.001502, mean≈0.000799
  - Disk_wr_kbs: global P95≈0.003829, P5≈0.002151 → anomaly_count_in_window=2, earliest=02:40, max≈0.004764, mean≈0.002711

- Observations / interpretation:
  - Many disk KPIs report anomaly_count = 30 (every minute in the 30‑minute window) and their earliest anomaly is 02:30 — i.e., sustained anomalies starting at the window start.
  - A large subset of disk KPIs have global P95 and P5 equal to 0 and in-window values equal to 0, which causes them to be flagged every minute (value >= P95 or <= P5). These are boundary-condition hits rather than clear high/low spikes.
  - Disk_svctm and Disk_await show substantive non-zero activity: Disk_svctm max=1.5 and Disk_await max=2.5 with many anomaly minutes, indicating elevated service time / I/O latency on os_006 during the incident window.
  - Disk_avgqu_sz and Disk_wr_kbs show a few brief anomaly minutes later in the window (02:36 and 02:40), indicating occasional queue/throughput events.

- Recommended next steps:
  - Investigate os_006 I/O subsystem and host logs around 02:30 for causes of sustained I/O anomalies (e.g., heavy I/O workload, device contention, degraded disk performance).
  - Correlate Disk_await and Disk_svctm spikes with db_006 DB activity (Physical_Read_Per_Sec, DF/LF writes) already flagged for anomalies.
  - Review instrumentation: many zero-valued KPIs with zero thresholds produce repeated boundary matches; verify data quality/normalization to avoid false-positives from zero-threshold conditions.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         kpi_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0   os_006      Disk_rd_kbs    0.000000   0.000000                       30    2020-05-30 02:30:00 CST             0.000000             0.000000              0.000000
1   os_006       Disk_svctm    0.000000   0.000000                       30    2020-05-30 02:30:00 CST             1.500000             0.000000              0.050000
2   os_006     Disk_io_util    0.000000   0.000000                       30    2020-05-30 02:30:00 CST             0.000000             0.000000              0.000000
3   os_006      Disk_wr_ios    0.000000   0.000000                       30    2020-05-30 02:30:00 CST             0.000000             0.000000              0.000000
4   os_006      Disk_rd_ios    0.000000   0.000000                       30    2020-05-30 02:30:00 CST             0.000000             0.000000              0.000000
5   os_006       Disk_await    1.338500   0.000000                       20    2020-05-30 02:30:00 CST             2.500000             0.000000              0.527667
6   os_006    Disk_avgqu_sz    0.001660   0.000497                        3    2020-05-30 02:36:00 CST             0.001502             0.000331              0.000799
7   os_006      Disk_wr_kbs    0.003829   0.002151                        2    2020-05-30 02:40:00 CST             0.004764             0.002068              0.002711
8   os_006     Buffers_used         NaN        NaN                        0                       <NA>             0.180965             0.180965              0.180965
9   os_006       Agent_ping         NaN        NaN                        0                       <NA>             1.000000             1.000000              1.000000
10  os_006     CPU_util_pct         NaN        NaN                        0                       <NA>             0.294559             0.202857              0.244999
11  os_006       Cache_used         NaN        NaN                        0                       <NA>             0.702213             0.702198              0.702206
12  os_006  CPU_system_time         NaN        NaN                        0                       <NA>             0.150075             0.102832              0.120148
13  os_006    CPU_user_time         NaN        NaN                        0                       <NA>             0.155620             0.102809              0.121168
14  os_006  CPU_iowait_time         NaN        NaN                        0                       <NA>             0.008339             0.000000              0.003428
15  os_006     CPU_idle_pct         NaN        NaN                        0                       <NA>            99.797143            99.705441             99.755072
16  os_006     FS_max_avail         NaN        NaN                        0                       <NA>            30.520500            30.520500             30.520500
17  os_006      FS_max_util         NaN        NaN                        0                       <NA>            16.000000            16.000000             16.000000
18  os_006   FS_total_space         NaN        NaN                        0                       <NA>            39.212100            39.212100             39.212100
19  os_006      FS_used_pct         NaN        NaN                        0                       <NA>            15.121200            15.121200             15.121200```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz

# reuse metric_df and tz if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Parameters
kp_list = ['Physical_Read_Per_Sec','DFParaWrite_Per_Sec','LFParaWrite_Per_Sec']
start_ts = 1590777000
end_ts   = 1590778800

# 1) Identify db_* rows for the KPIs (global scope)
mask_db_kpis = metric_df['cmdb_id'].astype(str).str.startswith('db_') & metric_df['kpi_name'].isin(kp_list)
global_pairs = metric_df[mask_db_kpis].copy()

# If none present, return explicit message
if global_pairs.shape[0] == 0:
    result = "No DB I/O KPIs found for db_* in metric.csv."
else:
    # 2) Compute global P95 and P5 per (cmdb_id,kpi_name) using full series (before filtering)
    thresholds = (
        global_pairs
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(global_P95=lambda x: x.quantile(0.95), global_P5=lambda x: x.quantile(0.05))
        .reset_index()
    )

    # 3) Filter metric.csv to the incident window and relevant pairs
    window_mask = (metric_df['timestamp'] >= start_ts) & (metric_df['timestamp'] <= end_ts)
    window_df = metric_df[window_mask & metric_df['cmdb_id'].astype(str).str.startswith('db_') & metric_df['kpi_name'].isin(kp_list)].copy()

    # 4) Merge thresholds into window and flag anomalies
    w = window_df.merge(thresholds, on=['cmdb_id','kpi_name'], how='right')  # right to keep pairs even if no window rows
    # Ensure value column present for rows without window data
    # For rows without window observations, value will be NaN
    w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

    # anomaly_count per pair (will be 0 if no window rows)
    anomaly_counts = (
        w.groupby(['cmdb_id','kpi_name'], as_index=False)['is_anomaly']
        .sum()
        .rename(columns={'is_anomaly':'anomaly_count_in_window'})
    )

    # earliest anomaly timestamp per pair (from rows where is_anomaly True)
    earliest = (
        w[w['is_anomaly']]
        .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
        .min()
        .rename(columns={'timestamp':'earliest_anomaly_epoch'})
    )

    # max and mean value in window per pair (for pairs with no window rows, these will be NaN)
    stats = (
        w.groupby(['cmdb_id','kpi_name'], as_index=False)['value']
        .agg(max_value_in_window='max', mean_value_in_window='mean')
    )

    # 5) Merge pieces together
    merged = thresholds.merge(anomaly_counts, on=['cmdb_id','kpi_name'], how='left') \
                       .merge(earliest, on=['cmdb_id','kpi_name'], how='left') \
                       .merge(stats, on=['cmdb_id','kpi_name'], how='left')

    # Format earliest timestamp to UTC+8 strings
    def fmt_epoch(e):
        if pd.isna(e):
            return pd.NA
        return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

    merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_epoch'].apply(fmt_epoch)

    # Round numeric columns for compactness
    for col in ['global_P95','global_P5','max_value_in_window','mean_value_in_window']:
        if col in merged.columns:
            merged[col] = merged[col].astype(float).round(6)

    # Select requested columns and sort
    out_cols = ['cmdb_id','kpi_name','global_P95','global_P5','anomaly_count_in_window',
                'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window']

    result_df = merged[out_cols].fillna({'anomaly_count_in_window':0}).copy()
    # Ensure anomaly_count is integer
    result_df['anomaly_count_in_window'] = result_df['anomaly_count_in_window'].astype(int)

    result = result_df.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

# Display final result (DataFrame or message)
result

```
Out[9]:
```
```python
# Summarize DB I/O KPI anomaly scan results in plain English (compact)
try:
    df = result  # reuse DataFrame from previous step
except NameError:
    df = None

if df is None or isinstance(df, str):
    summary = "No DB I/O KPI results available to summarize."
else:
    import numpy as np
    # basic stats
    n_rows = int(df.shape[0])
    total_anomaly_points = int(df['anomaly_count_in_window'].astype(int).sum())
    # top 5 by anomaly_count then max_value
    top5 = df.sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).head(5)
    top5_list = []
    for r in top5.itertuples(index=False):
        top5_list.append(f"{r.cmdb_id}/{r.kpi_name} count={int(r.anomaly_count_in_window)} max={float(r.max_value_in_window):.6g} mean={float(r.mean_value_in_window):.6g} P95={float(r.global_P95):.6g}")
    # notable extreme max values
    high_max = df.sort_values('max_value_in_window', ascending=False).head(3)
    high_list = [f"{r.cmdb_id}/{r.kpi_name} max={float(r.max_value_in_window):.6g} (mean={float(r.mean_value_in_window):.6g})" for r in high_max.itertuples(index=False)]
    # earliest anomaly times examples
    earliest_examples = df['earliest_anomaly_timestamp'].dropna().unique()[:3].tolist()

    summary = (
        "DB I/O KPI anomaly summary (Physical_Read_Per_Sec, DFParaWrite_Per_Sec, LFParaWrite_Per_Sec) "
        "for db_* during the incident window 2020-05-30 02:30–03:00 (UTC+8):\n"
        f"- Rows returned: {n_rows}. Total anomaly points (sum of counts): {total_anomaly_points}.\n"
        f"- Top candidates (examples):\n    " + "\n    ".join(top5_list) + "\n"
        f"- Notable highest peaks (examples): {', '.join(high_list)}.\n"
        f"- Earliest anomaly timestamps (examples): {', '.join(earliest_examples)}.\n"
        "- Observations:\n"
        "  * Several DBs show anomalies at every minute of the window (anomaly_count = 30), indicating sustained I/O activity or boundary hits.\n"
        "  * Many DF/LF write metrics and Physical_Read_Per_Sec have P5 = 0 (historical lower bounds), so lower-bound matches can contribute to counts.\n"
        "  * Some DBs exhibit very large Physical_Read_Per_Sec peaks (e.g., db_009 and db_003), suggesting heavy read I/O on those nodes during the window.\n"
        "- Recommendation: prioritize investigation of DBs with both high anomaly_count and large max read/write values (examples above). "
        "Correlate with host I/O metrics, disk latency (os_*), and DB workload (transactions, log writes) around 02:30 for root-cause analysis."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id               kpi_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0   db_010  Physical_Read_Per_Sec    117.8000       0.00                       30    2020-05-30 02:30:00 CST               119.80             11.846667
1   db_009    DFParaWrite_Per_Sec      0.0000       0.00                       30    2020-05-30 02:30:00 CST                 0.00              0.000000
2   db_013  Physical_Read_Per_Sec    125.2000       0.00                       29    2020-05-30 02:30:00 CST               125.20             12.516667
3   db_008  Physical_Read_Per_Sec    334.9500       0.00                       28    2020-05-30 02:30:00 CST               339.90             33.633333
4   db_012  Physical_Read_Per_Sec    125.1050       0.00                       28    2020-05-30 02:30:00 CST               127.20             12.580000
5   db_002  Physical_Read_Per_Sec      0.2050       0.00                       28    2020-05-30 02:30:00 CST                16.60              0.573333
6   db_001  Physical_Read_Per_Sec    313.2000       0.00                       27    2020-05-30 02:30:00 CST               317.70             31.466667
7   db_006  Physical_Read_Per_Sec     71.3050       0.00                       27    2020-05-30 02:30:00 CST                72.30              7.253333
8   db_004  Physical_Read_Per_Sec    302.4200       0.00                       26    2020-05-30 02:30:00 CST               317.60             30.746667
9   db_005  Physical_Read_Per_Sec    197.7300       0.00                       26    2020-05-30 02:30:00 CST               203.10             19.936667
10  db_011  Physical_Read_Per_Sec    117.8000       0.00                       26    2020-05-30 02:30:00 CST               119.70             11.856667
11  db_007  Physical_Read_Per_Sec     23.9050       0.00                       25    2020-05-30 02:30:00 CST                31.20              2.650000
12  db_006    DFParaWrite_Per_Sec      1.3990       0.00                       24    2020-05-30 02:30:00 CST                 1.83              0.301667
13  db_009  Physical_Read_Per_Sec   2326.4650       0.00                       23    2020-05-30 02:30:00 CST              2525.80            284.876667
14  db_005    DFParaWrite_Per_Sec      3.9110       0.00                       22    2020-05-30 02:30:00 CST                14.69              0.891333
15  db_006    LFParaWrite_Per_Sec      1.7160       0.00                       21    2020-05-30 02:31:00 CST                 1.86              0.554667
16  db_012    DFParaWrite_Per_Sec      0.0705       0.05                       18    2020-05-30 02:30:00 CST                 0.12              0.060000
17  db_001    DFParaWrite_Per_Sec      2.9805       0.00                       12    2020-05-30 02:30:00 CST                 6.62              1.116333
18  db_011    LFParaWrite_Per_Sec      0.2200       0.06                       10    2020-05-30 02:35:00 CST                 0.25              0.112333
19  db_003  Physical_Read_Per_Sec   2524.0450       0.10                        8    2020-05-30 02:31:00 CST              2925.40            834.123333```
```

In [ ]:
# In[10]:

import pandas as pd
import pytz

# Reuse existing metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Parameters
kpis = ['Sess_Active','Session_pct','tnsping_result_time','On_Off_State','Hang',
        'rejected_connections','Login_Per_Sec','User_Commit']
start_ts = 1590777000
end_ts = 1590778800

# 1) Scope rows for db_* and the KPI list (for threshold computation)
db_mask = metric_df['cmdb_id'].astype(str).str.startswith('db_')
db_all = metric_df[db_mask].copy()

# Check presence of any of the target KPIs among db_*
db_kpi_rows = db_all[db_all['kpi_name'].isin(kpis)].copy()
if db_kpi_rows.shape[0] == 0:
    result = "No connection/session KPIs found for db_* in metric.csv."
else:
    # 2) Compute global P95 and P5 per (cmdb_id,kpi_name) using full historical series (before window filter)
    thresholds = (
        db_kpi_rows
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(global_P95=lambda x: x.quantile(0.95),
             global_P5 =lambda x: x.quantile(0.05))
        .reset_index()
    )

    # 3) Filter metric.csv to the incident window for db_* and KPIs
    window = db_all[
        (db_all['timestamp'] >= start_ts) &
        (db_all['timestamp'] <= end_ts) &
        (db_all['kpi_name'].isin(kpis))
    ].copy()

    # If no window rows for these KPIs
    if window.shape[0] == 0:
        result = "No connection/session KPIs found for db_* in metric.csv."
    else:
        # 4) Merge thresholds and flag anomalies (value >= P95 OR value <= P5)
        merged = window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        merged['is_anomaly'] = ((merged['value'] >= merged['global_P95']) | (merged['value'] <= merged['global_P5'])).fillna(False)

        # Aggregate per (cmdb_id, kpi_name)
        anomaly_counts = (
            merged.groupby(['cmdb_id','kpi_name'], as_index=False)['is_anomaly']
            .sum()
            .rename(columns={'is_anomaly':'anomaly_count_in_window'})
        )

        earliest = (
            merged[merged['is_anomaly']]
            .groupby(['cmdb_id','kpi_name'], as_index=False)['timestamp']
            .min()
            .rename(columns={'timestamp':'earliest_anomaly_epoch'})
        )

        stats = (
            merged.groupby(['cmdb_id','kpi_name'], as_index=False)['value']
            .agg(max_value_in_window='max', mean_value_in_window='mean')
        )

        # Join thresholds (to include P95/P5 even if anomaly_count is 0)
        agg = thresholds.merge(anomaly_counts, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(earliest, on=['cmdb_id','kpi_name'], how='left') \
                        .merge(stats, on=['cmdb_id','kpi_name'], how='left')

        # Fill NaNs for counts with 0
        agg['anomaly_count_in_window'] = agg['anomaly_count_in_window'].fillna(0).astype(int)

        # Format earliest timestamp to UTC+8 human string
        def fmt_epoch(e):
            if pd.isna(e):
                return pd.NA
            return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

        agg['earliest_anomaly_timestamp'] = agg['earliest_anomaly_epoch'].apply(fmt_epoch)

        # Round numeric columns for compactness
        for col in ['global_P95','global_P5','max_value_in_window','mean_value_in_window']:
            if col in agg.columns:
                agg[col] = agg[col].astype(float).round(6)

        # 5) Select requested columns and sort by anomaly_count desc then max_value desc
        out_cols = ['cmdb_id','kpi_name','global_P95','global_P5','anomaly_count_in_window',
                    'earliest_anomaly_timestamp','max_value_in_window','mean_value_in_window']

        result_df = agg[out_cols].sort_values(['anomaly_count_in_window','max_value_in_window'], ascending=[False, False]).head(20).reset_index(drop=True)

        # If none of these KPIs present (defensive), return explicit message
        if result_df.shape[0] == 0:
            result = "No connection/session KPIs found for db_* in metric.csv."
        else:
            result = result_df

# Display the result (DataFrame or message)
result

```
Out[10]:
```
- Scope: scanned connection/session KPIs for all db_* during incident window 2020-05-30 02:30:00–03:00:00 (UTC+8).

- Key findings (top rows):
  - tnsping_result_time: multiple DBs show sustained breaches (anomaly_count = 30)
    - db_010, db_011, db_012, db_013 — global_P95=20, global_P5=10, max=20, earliest anomaly 2020-05-30 02:30:00, mean ~16–18 ms.
    - db_002, db_004, db_005, db_006, db_007 — global_P95 either 10 or 20, all anomaly_count = 30, earliest 02:30.
  - Login_Per_Sec: high sustained counts
    - db_003: P95=1.2, anomaly_count=30, max=2.8, mean≈1.23
    - db_002: P95=0.4, anomaly_count=30, max=2.3, mean≈0.45
  - Sess_Active / Session_pct:
    - db_009 Sess_Active: P95=1.0, anomaly_count=30, max=1.1
  - Hang and On_Off_State: many DBs (db_001, db_002, db_003, db_004, etc.) show constant value 1.0, anomaly_count=30 (breaches every minute because value equals P95/P5).
  - Timing: for most flagged metrics earliest anomaly = 2020-05-30 02:30:00 — anomalies start at the window boundary and persist through the window.

- Interpretation:
  - Connection/heartbeat latency (tnsping_result_time) is elevated and sustained across several DB nodes — this indicates network or database responsiveness degradation affecting multiple DBs.
  - Login and session metrics show elevated activity on some nodes (e.g., db_003, db_002) concurrently.
  - Many KPIs are constant at boundary values (Hang, On_Off_State), which produces repeated anomaly flags — this may reflect a sustained state (e.g., service hung/on) or threshold boundary behavior.
  - The pattern is systemic (starts at 02:30 across many DBs) rather than isolated spikes.

- Actionable suggestions:
  - Immediately inspect network connectivity and latency between clients and the affected DB nodes (start with db_010/011/012/013 and db_002/003).
  - Check DB process health, listeners, and acceptor threads for login/session saturation on db_002 and db_003.
  - Correlate with host I/O and OS metrics (disk latency, CPU, network) and with trace-level errors to narrow root cause.
  - Verify instrumentation/thresholds for KPIs that are constant at boundary values to reduce false-positive noise.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id             kpi_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  mean_value_in_window
0   db_010  tnsping_result_time        20.0       10.0                       30    2020-05-30 02:30:00 CST                 20.0             18.000000
1   db_011  tnsping_result_time        20.0       10.0                       30    2020-05-30 02:30:00 CST                 20.0             16.000000
2   db_012  tnsping_result_time        20.0       10.0                       30    2020-05-30 02:30:00 CST                 20.0             17.666667
3   db_013  tnsping_result_time        20.0       10.0                       30    2020-05-30 02:30:00 CST                 20.0             18.000000
4   db_002  tnsping_result_time        10.0        0.0                       30    2020-05-30 02:30:00 CST                 10.0              1.666667
5   db_004  tnsping_result_time        10.0        0.0                       30    2020-05-30 02:30:00 CST                 10.0              3.000000
6   db_005  tnsping_result_time        10.0        0.0                       30    2020-05-30 02:30:00 CST                 10.0              3.000000
7   db_006  tnsping_result_time        10.0        0.0                       30    2020-05-30 02:30:00 CST                 10.0              1.333333
8   db_007  tnsping_result_time        10.0        0.0                       30    2020-05-30 02:30:00 CST                 10.0              2.000000
9   db_003        Login_Per_Sec         1.2        1.1                       30    2020-05-30 02:30:00 CST                  2.8              1.233333
10  db_002        Login_Per_Sec         0.4        0.3                       30    2020-05-30 02:30:00 CST                  2.3              0.450000
11  db_009          Sess_Active         1.0        0.9                       30    2020-05-30 02:30:00 CST                  1.1              0.930000
12  db_001                 Hang         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
13  db_001         On_Off_State         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
14  db_002                 Hang         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
15  db_002         On_Off_State         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
16  db_003                 Hang         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
17  db_003         On_Off_State         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
18  db_004                 Hang         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000
19  db_004         On_Off_State         1.0        1.0                       30    2020-05-30 02:30:00 CST                  1.0              1.000000```
```

In [ ]:
# In[11]:

import pandas as pd
import pytz
import re

# reuse metric_df and tz if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv('metric.csv')
    metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
    metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

os_host = 'os_006'
start_ts = 1590777000
end_ts = 1590778800

# 1) Identify network-related KPI names for os_006 (case-insensitive search)
os_rows = metric_df[metric_df['cmdb_id'] == os_host].copy()
pattern = r'network|incoming|outgoing|recv|send|agent_ping|recv_total|send_total'
net_kpis = sorted(os_rows[os_rows['kpi_name'].str.contains(pattern, case=False, na=False)]['kpi_name'].unique().tolist())

if len(net_kpis) == 0:
    result = f"No network KPIs found for {os_host}."
else:
    # 2) Compute global P95 and P5 per (os_006, kpi_name) using full series (before window filtering)
    thresholds = (
        metric_df[(metric_df['cmdb_id'] == os_host) & (metric_df['kpi_name'].isin(net_kpis))]
        .groupby(['cmdb_id','kpi_name'])['value']
        .agg(global_P95=lambda x: x.quantile(0.95), global_P5=lambda x: x.quantile(0.05))
        .reset_index()
    )

    # 3) Filter to incident window for os_006 & network KPIs
    window_os = metric_df[
        (metric_df['cmdb_id'] == os_host) &
        (metric_df['timestamp'] >= start_ts) &
        (metric_df['timestamp'] <= end_ts) &
        (metric_df['kpi_name'].isin(net_kpis))
    ].copy()

    if window_os.shape[0] == 0:
        result = f"No metric data for {os_host} in the incident window {start_ts}..{end_ts}."
    else:
        # 4) Merge thresholds and flag anomalies (value >= P95 OR value <= P5)
        w = window_os.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
        w['is_anomaly'] = ((w['value'] >= w['global_P95']) | (w['value'] <= w['global_P5'])).fillna(False)

        anomaly_counts = w.groupby('kpi_name', as_index=False)['is_anomaly'].sum().rename(columns={'is_anomaly':'anomaly_count_in_window'})
        earliest = w[w['is_anomaly']].groupby('kpi_name', as_index=False)['timestamp'].min().rename(columns={'timestamp':'earliest_anomaly_epoch'})
        stats = w.groupby('kpi_name', as_index=False)['value'].agg(max_value_in_window='max', min_value_in_window='min', mean_value_in_window='mean')

        merged = thresholds.merge(anomaly_counts, on='kpi_name', how='right') \
                           .merge(earliest, on='kpi_name', how='left') \
                           .merge(stats, on='kpi_name', how='left')

        merged['cmdb_id'] = os_host

        def fmt_epoch(e):
            if pd.isna(e):
                return pd.NA
            return pd.to_datetime(int(e), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')

        merged['earliest_anomaly_timestamp'] = merged['earliest_anomaly_epoch'].apply(fmt_epoch)

        for col in ['global_P95','global_P5','max_value_in_window','min_value_in_window','mean_value_in_window']:
            if col in merged.columns:
                merged[col] = merged[col].astype(float).round(6)

        merged['anomaly_count_in_window'] = merged['anomaly_count_in_window'].fillna(0).astype(int)

        out_cols = ['cmdb_id','kpi_name','global_P95','global_P5','anomaly_count_in_window',
                    'earliest_anomaly_timestamp','max_value_in_window','min_value_in_window','mean_value_in_window']

        result = merged[out_cols].sort_values('anomaly_count_in_window', ascending=False).head(20).reset_index(drop=True)

# display result (DataFrame or message)
result

```
Out[11]:
```
- OS host: examined os_006 network KPIs.

- Network KPIs found: Agent_ping, Recv_total, Outgoing_network_traffic, Send_total, Incoming_network_traffic.

- Key results (incident window 2020-05-30 02:30–03:00 UTC+8):
  - Agent_ping: global P95=1.0, P5=1.0 — anomaly_count=30 (every minute), earliest 2020-05-30 02:30:00, values constant 1.0.  
    - Interpretation: ping status constant at its threshold -> sustained boundary condition (not necessarily a transient network outage).
  - Recv_total: global P95≈0.668, P5≈0.488 — anomaly_count=4, earliest 02:33, max≈1.069, mean≈0.567.  
    - Interpretation: a few short receive bursts starting ~02:33.
  - Outgoing_network_traffic: global P95≈0.000138, P5≈0.000104 — anomaly_count=3, earliest 02:30, max≈0.000139, mean≈0.000122.  
  - Send_total: global P95≈0.000473, P5≈0.000307 — anomaly_count=3, earliest 02:32, max≈0.000479, mean≈0.000369.  
  - Incoming_network_traffic: global P95≈0.000209, P5≈0.000158 — anomaly_count=1 at 02:46, max≈0.000211, mean≈0.000181.

- Overall interpretation:
  - Agent_ping shows a sustained boundary value (1.0) across the window — indicates persistent ping state (contact alive or constant ping result) rather than intermittent loss.
  - Network traffic metrics show a few brief anomalies (small bursts in send/recv) at specific times (02:30–02:46) but not sustained across the whole window like disk/I/O and DB KPIs.
  - Magnitudes are small for incoming/outgoing traffic in normalized units; Recv_total has a few notable peaks.

- Suggested follow-up:
  - Correlate the small network bursts (Recv_total, Send_total, Incoming_network_traffic) timestamps with the DB I/O and trace anomalies (around 02:30–02:40) to see if network jitter contributed to DB latency.
  - Verify what Agent_ping=1.0 represents in instrumentation (healthy/unhealthy code) to avoid misinterpreting repeated threshold matches as transient network errors.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                  kpi_name  global_P95  global_P5  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  min_value_in_window  mean_value_in_window
0  os_006                Agent_ping    1.000000   1.000000                       30    2020-05-30 02:30:00 CST             1.000000             1.000000              1.000000
1  os_006                Recv_total    0.668033   0.487649                        4    2020-05-30 02:33:00 CST             1.069195             0.366183              0.566625
2  os_006  Outgoing_network_traffic    0.000138   0.000104                        3    2020-05-30 02:30:00 CST             0.000139             0.000102              0.000122
3  os_006                Send_total    0.000473   0.000307                        3    2020-05-30 02:32:00 CST             0.000479             0.000303              0.000369
4  os_006  Incoming_network_traffic    0.000209   0.000158                        1    2020-05-30 02:46:00 CST             0.000211             0.000158              0.000181```
```